# XGBoost - Cross Validation (WDBC)

Notebook para executar XGBoost com StratifiedKFold sobre o dataset WDBC.
- Selecione qual dataset normalizado usar alterando a variável `DATASET_PATH` no bloco de configuração.
- Os parâmetros do modelo estão em `xgb_params`.
- Outputs: `model_reports/xgboost/cv/` contém `csv/`, `images/`, `pdf/`.
- O modelo será salvo em `databases/WDBC/common/models/xgboost/v1/xgb_model_cf{N_SPLITS}.pkl`.

In [1]:
# Configuração
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = Path('../../')  # ajustável se necessário
# Escolha um dos 3 datasets normalizados gerados pelo EDA: 'Standard', 'Robust', 'MinMax'
DATASET_NAME = 'exams.csv'  # alterar para exams.csv ou exams_robust_scaled.csv ou exams_minmax_scaled.csv
DATASET_PATH = Path('../data/processed') / DATASET_NAME
# Se quiser forçar explicitamente qual coluna é o target (no bruto ou no normalizado), defina aqui
# Exemplo: TARGET_COLUMN = 'diagnosis' ou TARGET_COLUMN = 'y'
TARGET_COLUMN = 'septic'  # ajustar se necessário

# Output paths
REPORTS = Path('../model_reports/xgboost/cv')
CSV_OUT = REPORTS / 'csv'
IMAGES_OUT = REPORTS / 'images'
PDF_OUT = REPORTS / 'pdf'
MODEL_DIR = Path('../common/models/xgboost/v1')
# Basico outputs (single train/test) - sibling folder to 'cv'
BASICO_DIR = REPORTS.parent / 'basico'
BASICO_CSV = BASICO_DIR / 'csv'
BASICO_IMAGES = BASICO_DIR / 'images'

for d in [REPORTS, CSV_OUT, IMAGES_OUT, PDF_OUT, MODEL_DIR, BASICO_DIR, BASICO_CSV, BASICO_IMAGES]:
    d.mkdir(parents=True, exist_ok=True)

# XGBoost params (variável conforme solicitado)
xgb_params = {
    "n_estimators": 650,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "gamma": 0.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "min_child_weight": 1,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "n_jobs": 8,
    "random_state": 42
}

# CV and threshold config
N_SPLITS = 10
THRESHOLD = 0.5
# Colunas identificadoras que NÃO devem entrar como features no modelo, mas devem ser mantidas nos arquivos de saída
#EXCLUDE_COLUMNS = ['id','ID', 'patient_id', 'pid','septic','stay_id']
#EXCLUDE_COLUMNS = ['id','ID', 'patient_id', 'pid','septic','stay_id', 'Chloride (serum)' , 'Creatinine (serum)', 'race_white', 'race_unknown', 'race_black', 'race_hispanic/latino', 'race_other','gender_M']
EXCLUDE_COLUMNS = ['id','ID', 'patient_id', 'pid','septic','stay_id', 'race_white', 'race_unknown', 'race_black', 'race_hispanic/latino', 'race_other']
# Pareto configuration: threshold (fraction) and selected-features placeholder (set to None to be updated after CV)
PARETO_THRESHOLD = 0.9  # fraction, e.g. 0.8 == 80%
PARETO_SELECTED_FEATURES = None

# Variance percent-difference threshold (fraction). Example: 0.1 == 10%% difference.
VAR_DIFF_PCT_THRESHOLD = 0.01
TARGET_COLUMN = 'septic'  # ajustar se necessário
TEST_SIZE = 0.3
RT_RANDOM_STATE = 42


In [2]:
# Imports principais
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
                             balanced_accuracy_score, confusion_matrix, roc_curve, log_loss, auc)
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm
import joblib

sns.set(style='whitegrid')

In [3]:
# Load dataset (path definido na configuração)
import shutil
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Arquivo de dataset não encontrado: {DATASET_PATH.resolve()}')

# Preserve a raw copy of the original CSV (unaltered) in data/processed with suffix _raw
RAW_COPY_PATH = Path('../data/processed') / f"{DATASET_NAME.replace('.csv','')}_raw.csv"
try:
    shutil.copy2(DATASET_PATH, RAW_COPY_PATH)
    print(f'Raw dataset copiado para: {RAW_COPY_PATH}')
except Exception as e_copy:
    print('Não foi possível copiar o CSV original para processed:', e_copy)

# Leia o dataset (continuamos a usar df para processamento posterior)
df = pd.read_csv(DATASET_PATH)
# Função utilitária para encontrar coluna aproximada (case/strip tolerant)
def find_column(df, name):
    # tenta match exato
    if name in df.columns:
        return name
    # tenta match case-insensitive
    low = {c.lower().strip(): c for c in df.columns}
    if name.lower().strip() in low:
        return low[name.lower().strip()]
    # tenta encontrar coluna que contenha o nome
    for c in df.columns:
        if name.lower().strip() in c.lower():
            return c
    return None

# Determina a coluna alvo: usa TARGET_COLUMN quando fornecida, senão infere
target_col = None
if TARGET_COLUMN:
    candidate = find_column(df, TARGET_COLUMN)
    if candidate is None:
        raise ValueError(f"TARGET_COLUMN='{TARGET_COLUMN}' não encontrada. Colunas: {list(df.columns)}")
    target_col = candidate
else:
    # heurística leve: nomes comuns
    for cand in  ['y', 'septic', 'target', 'label', 'class']:
        candidate = find_column(df, cand)
        if candidate:
            target_col = candidate
            break
    # procura colunas binárias
    if target_col is None:
        for c in df.columns:
            if c.lower().strip() in ('id', 'patient_id', 'pid', 'septic', 'stay_id'):
                continue
            try:
                nunq = df[c].nunique(dropna=True)
            except Exception:
                nunq = 999
            if nunq == 2:
                target_col = c
                break
    # fallback para última coluna com poucos valores distintos
    if target_col is None:
        lastc = df.columns[-1]
        if df[lastc].nunique(dropna=True) <= 10:
            target_col = lastc
    if target_col is None:
        raise ValueError(f"Não foi possível inferir a coluna alvo. Defina TARGET_COLUMN ou verifique as colunas: {list(df.columns)}")

# Renomeia para 'y' e garante coluna disponível antes de acessar
if target_col != 'y':
    df = df.rename(columns={target_col: 'y'})
if 'y' not in df.columns:
    raise KeyError('Coluna y não encontrada após renomeação')

# Se y não for numérico, discretiza (M/B -> 1/0 ou factorize para múltiplas classes)
if not np.issubdtype(df['y'].dtype, np.number):
    unique_vals = list(df['y'].dropna().unique())
    low = [str(u).lower() for u in unique_vals]
    if set(low) <= set(['0', '1']):
        mapping = {v: (1 if str(v).lower() == '1' else 0) for v in unique_vals}
        df['y'] = df['y'].map(mapping)
    else:
        # factorize para inteiros 0..k-1
        df['y'], uniques = pd.factorize(df['y'])
    # salva versão discretizada para inspeção
    df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_y_numeric.csv', index=False)

# Substitui inf valores e dropna (pode ajustar imputação se preferir)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna()

# reorganiza para garantir que y seja a última coluna
cols = [c for c in df.columns if c != 'y'] + ['y']
df = df[cols]

# salva uma cópia de entrada no csv de reports (final)
df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}', index=False)

print('Dataset shape:', df.shape)

Raw dataset copiado para: ..\data\processed\exams_raw.csv
Dataset shape: (94446, 33)


In [8]:
# ============================
# XGBoost 8/1/1 com Optuna (TPE + Pruning) + métricas e CSVs — alinhado ao RF
# AJUSTADO (mesma lógica dos anteriores):
# 1) Split 8/1/1 hold-out CORRETO (val/test fora do Optuna)
# 2) Optuna otimiza por AUC (default) ou ACC (parametrizável)
# 3) Threshold AUTOMÁTICO na validação hold-out (robusto) para otimizar F1/ACC/BAC
#    - evita degeneração "tudo 0" via MIN_POS_PRED (fração mínima de positivos previstos)
# 4) Aplica threshold fixo no teste hold-out
# 5) Salva CSVs long + CSV final (val/test no formato da sua tabela)
# ============================

import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score
)
import joblib
from pathlib import Path
import json

# -------------------- CONFIG --------------------
RANDOM_STATE = 42
N_JOBS = 8
EARLY_STOP_ROUNDS = 100
EPS = 1e-12

# Optuna: "auc" (default) ou "accuracy"
OPTIMIZE_METRIC = 'auc'
OPTIMIZE_METRIC = globals().get("OPTIMIZE_METRIC", "auc")
OPTIMIZE_METRIC = str(OPTIMIZE_METRIC).lower().strip()
if OPTIMIZE_METRIC not in {"auc", "accuracy"}:
    raise ValueError("OPTIMIZE_METRIC deve ser 'auc' ou 'accuracy'.")

THRESH_OPTIMIZE_FOR = 'f1'
# Threshold: otimiza "f1" (default), "accuracy" ou "bac"
THRESH_OPTIMIZE_FOR = globals().get("THRESH_OPTIMIZE_FOR", "f1")
THRESH_OPTIMIZE_FOR = str(THRESH_OPTIMIZE_FOR).lower().strip()
if THRESH_OPTIMIZE_FOR not in {"f1", "accuracy", "bac"}:
    raise ValueError("THRESH_OPTIMIZE_FOR deve ser 'f1', 'accuracy' ou 'bac'.")

# grade e limites (robusto)
THRESH_GRID_STEP = 0.001
THRESH_GRID_STEP = float(globals().get("THRESH_GRID_STEP", 0.001))
THRESH_MIN = float(globals().get("THRESH_MIN", 0.05))
THRESH_MAX = float(globals().get("THRESH_MAX", 0.95))
if not (0.0 < THRESH_MIN < THRESH_MAX < 1.0):
    raise ValueError("THRESH_MIN/THRESH_MAX devem satisfazer 0 < min < max < 1.")

# Evita F1=0 por "tudo 0": mínima fração (ou contagem) de positivos previstos
# - se <1: interpretado como FRAÇÃO do conjunto (ex.: 0.01 = pelo menos 1% positivos previstos)
# - se >=1: interpretado como CONTAGEM mínima de positivos previstos
MIN_POS_PRED = globals().get("MIN_POS_PRED", 0.01)

# Optuna trials
N_TRIALS_OPTUNA = 5
N_TRIALS_OPTUNA = int(globals().get("N_TRIALS_OPTUNA", 5))

# Saídas
CSV_OUT = globals().get("CSV_OUT", None)
MODEL_DIR = globals().get("MODEL_DIR", None)

# -------------------- Checagens --------------------
try:
    df
except NameError:
    raise RuntimeError("df não está definido. Rode a célula de carregamento antes.")

try:
    EXCLUDE_COLUMNS
except NameError:
    EXCLUDE_COLUMNS = ['__cls_tmp__', 'orig_index', 'diagnosis', 'id', 'ID', 'patient_id', 'pid',
                       'fold', 'y_train', 'prob_0', 'prob_1', 'y_proba', 'y_pred']

if 'y' not in df.columns:
    raise RuntimeError("Coluna 'y' não encontrada em df.")

print(f"🎯 Optuna otimizando por: {OPTIMIZE_METRIC.upper()}")
print(f"🎚️ Threshold robusto otimiza: {THRESH_OPTIMIZE_FOR.upper()} (validação hold-out)")
print(f"🔒 Threshold range: [{THRESH_MIN:.2f}, {THRESH_MAX:.2f}] | step={THRESH_GRID_STEP:.2f}")
print(f"🧷 MIN_POS_PRED: {MIN_POS_PRED} (fração ou inteiro)")

# -------------------- Monta X / y --------------------
y_raw = df['y'].copy()

exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower and c != 'y']
FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]

X_model = df[FEATURES].copy()
X_model.replace([np.inf, -np.inf], np.nan, inplace=True)
X_model = X_model.fillna(X_model.median(numeric_only=True))

classes = np.unique(y_raw.dropna().values)
n_classes = len(classes)
is_binary = (n_classes == 2)

if not is_binary:
    raise RuntimeError("Este script (com threshold) está configurado para classificação binária.")

# mapeia para 0/1 de forma estável (evita classes {1,2} etc.)
class_to_bin = {classes[0]: 0, classes[1]: 1}
y = y_raw.map(class_to_bin).astype(int)

neg = int((y == 0).sum())
pos = int((y == 1).sum())
scale_pos_weight_default = float(neg / max(pos, 1))

print(f"✅ Dados: X={X_model.shape} | y={y.shape} | classes={classes} -> (0/1)")
print(f"✅ pos={pos} | neg={neg} | scale_pos_weight={scale_pos_weight_default:.3f}")

# ===================== 8/1/1: folds externos =====================
outer_kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(outer_kf.split(X_model, y))

test1_idx = fold_indices[-1][1]   # fold 10 (teste hold-out)
val1_idx  = fold_indices[-2][1]   # fold 9  (val hold-out)
train8_idx = np.setdiff1d(np.arange(len(X_model)), np.concatenate([val1_idx, test1_idx]))

X_train8, y_train8 = X_model.iloc[train8_idx], y.iloc[train8_idx]
X_val1,   y_val1   = X_model.iloc[val1_idx],   y.iloc[val1_idx]
X_test1,  y_test1  = X_model.iloc[test1_idx],  y.iloc[test1_idx]

print("\n=== Split 8/1/1 (hold-out) — CORRETO ===")
print(f"Treino (8/10): {len(train8_idx)} | Validação hold-out (1/10): {len(val1_idx)} | Teste hold-out (1/10): {len(test1_idx)}")
print("✅ Validação e Teste NÃO entram no Optuna.\n")

# ===================== Optuna (CV interna = 8 folds no treino) =====================
cv_for_optuna = StratifiedKFold(n_splits=8, shuffle=True, random_state=RANDOM_STATE)

def _safe_auc(y_true, p_pos):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, p_pos))

def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators":        trial.suggest_int("n_estimators", 400, 800),
        "max_depth":           trial.suggest_int("max_depth", 6, 12),
        "learning_rate":       trial.suggest_float("learning_rate", 0.005, 0.08, log=True),
        "subsample":           trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree":    trial.suggest_float("colsample_bytree", 0.5, 0.9),
        "min_child_weight":    trial.suggest_int("min_child_weight", 3, 10),
        "gamma":               trial.suggest_float("gamma", 0.0, 8.0),
        "reg_alpha":           trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
        "reg_lambda":          trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "max_delta_step":      trial.suggest_int("max_delta_step", 0, 6),

        "tree_method":         "hist",
        "n_jobs":              N_JOBS,
        "random_state":        RANDOM_STATE,
        "verbosity":           0,

        "objective":           "binary:logistic",
        "eval_metric":         "auc",
        "scale_pos_weight":    scale_pos_weight_default,
    }

    maximize_metric = True
    fold_scores = []

    for fold_id, (tr_idx, va_idx) in enumerate(cv_for_optuna.split(X_train8, y_train8), start=1):
        X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
        y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

        model = XGBClassifier(**params)

        use_callbacks = hasattr(xgb, "callback") and hasattr(xgb.callback, "EarlyStopping")
        if use_callbacks:
            es_cb = xgb.callback.EarlyStopping(
                rounds=EARLY_STOP_ROUNDS,
                save_best=True,
                maximize=maximize_metric
            )
            try:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[es_cb], verbose=False)
            except TypeError:
                model.fit(X_tr, y_tr)
        else:
            try:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            except TypeError:
                model.fit(X_tr, y_tr)

        p_va = model.predict_proba(X_va)[:, 1]

        if OPTIMIZE_METRIC == "auc":
            s = _safe_auc(y_va, p_va)
            if np.isnan(s):
                s = 0.5
        else:
            y_hat = (p_va >= 0.5).astype(int)  # fixo na CV interna
            s = float(accuracy_score(y_va, y_hat))

        fold_scores.append(float(s))
        trial.report(float(np.mean(fold_scores)), fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))

sampler = TPESampler(seed=RANDOM_STATE, multivariate=True, group=True)
pruner  = MedianPruner(n_warmup_steps=2)

study = optuna.create_study(
    study_name=f"xgb_tpe_cv811_{OPTIMIZE_METRIC}",
    direction="maximize",
    sampler=sampler,
    pruner=pruner
)
study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)

print(f"\n🏆 Melhor score ({OPTIMIZE_METRIC.upper()}, CV interna 8 folds): {study.best_value:.6f}")
print("Best trial:", study.best_trial.number)

best = study.best_trial.params.copy()

# --------- Dicionário final para reuso ---------
xgb_params = {
    "n_estimators":      int(best.get("n_estimators", 600)),
    "max_depth":         int(best.get("max_depth", 8)),
    "learning_rate":     float(best.get("learning_rate", 0.03)),
    "subsample":         float(best.get("subsample", 0.8)),
    "colsample_bytree":  float(best.get("colsample_bytree", 0.8)),
    "min_child_weight":  int(best.get("min_child_weight", 3)),
    "gamma":             float(best.get("gamma", 0.0)),
    "reg_alpha":         float(best.get("reg_alpha", 0.0)),
    "reg_lambda":        float(best.get("reg_lambda", 1.0)),
    "max_delta_step":    int(best.get("max_delta_step", 0)),

    "tree_method":       "hist",
    "n_jobs":            N_JOBS,
    "random_state":      RANDOM_STATE,
    "verbosity":         0,

    "objective":         "binary:logistic",
    "eval_metric":       "auc",
    "scale_pos_weight":  scale_pos_weight_default,
}

print("\n=== xgb_params finais (8/1/1) ===")
for k, v in xgb_params.items():
    print(f"{k} = {v}")

# ===================== Métricas (mesmo formato RF) =====================
def cross_entropy_per_class(y_true, p_pos, eps=1e-12):
    y_true = np.asarray(y_true).astype(int)
    p_pos = np.clip(np.asarray(p_pos), eps, 1 - eps)
    mask0 = (y_true == 0)
    mask1 = (y_true == 1)
    ce0 = float(np.mean(-np.log(1 - p_pos[mask0]))) if np.any(mask0) else np.nan
    ce1 = float(np.mean(-np.log(p_pos[mask1])))     if np.any(mask1) else np.nan
    return ce0, ce1

def all_metrics_table_long(y_true, proba_pos, threshold=0.5, eps=1e-12):
    y_true = np.asarray(y_true).astype(int)
    proba_pos = np.asarray(proba_pos)
    y_pred = (proba_pos >= threshold).astype(int)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN

    n0 = int(np.sum(y_true == 0))
    n1 = int(np.sum(y_true == 1))
    prop_c0 = n0 / total if total > 0 else np.nan
    prop_c1 = n1 / total if total > 0 else np.nan

    ce0, ce1 = cross_entropy_per_class(y_true, proba_pos, eps=eps)

    acc  = float(accuracy_score(y_true, y_pred))
    f1   = float(f1_score(y_true, y_pred, zero_division=0))
    prec = float(precision_score(y_true, y_pred, zero_division=0))
    rec  = float(recall_score(y_true, y_pred, zero_division=0))
    bac  = float(balanced_accuracy_score(y_true, y_pred))
    auc  = _safe_auc(y_true, proba_pos)

    TP_pct = TP / total if total > 0 else np.nan
    FP_pct = FP / total if total > 0 else np.nan
    TN_pct = TN / total if total > 0 else np.nan
    FN_pct = FN / total if total > 0 else np.nan

    return {
        "Acurácia": acc,
        "Cross-Entropy C0": ce0,
        "Proporção C0": prop_c0,
        "Cross-Entropy C1": ce1,
        "Proporção C1": prop_c1,
        "F1 Score": f1,
        "Balanced Accuracy": bac,
        "Precision": prec,
        "Recall": rec,
        "AUROC": auc,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "TP(%)": TP_pct, "FP(%)": FP_pct, "TN(%)": TN_pct, "FN(%)": FN_pct,
    }

def metrics_for_table(y_true, proba_pos, threshold=0.5, eps=1e-12):
    y_true = np.asarray(y_true).astype(int)
    proba_pos = np.asarray(proba_pos)
    y_pred = (proba_pos >= threshold).astype(int)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN

    acc  = float(accuracy_score(y_true, y_pred))
    f1   = float(f1_score(y_true, y_pred, zero_division=0))
    prec = float(precision_score(y_true, y_pred, zero_division=0))
    rec  = float(recall_score(y_true, y_pred, zero_division=0))
    bac  = float(balanced_accuracy_score(y_true, y_pred))
    auc  = _safe_auc(y_true, proba_pos)

    ce0, ce1 = cross_entropy_per_class(y_true, proba_pos, eps=eps)

    TP_pct = TP / total if total > 0 else np.nan
    FP_pct = FP / total if total > 0 else np.nan
    TN_pct = TN / total if total > 0 else np.nan
    FN_pct = FN / total if total > 0 else np.nan

    return {
        "ACC": acc, "F1": f1, "BAC": bac, "PRE": prec, "REC": rec, "AUC": auc,
        "CE-C0": ce0, "CE-C1": ce1, "CE-C2": "---", "y": 1,
        "TP%": TP_pct, "FP%": FP_pct, "TN%": TN_pct, "FN%": FN_pct
    }

# -------------------- Threshold robusto (evita F1=0 por tudo 0) --------------------
def find_best_threshold_robust(y_true, p_pos, optimize_for="f1",
                               tmin=0.05, tmax=0.95, step=0.01,
                               min_pos_pred=0.01):
    y_true = np.asarray(y_true).astype(int)
    p_pos  = np.asarray(p_pos)

    if min_pos_pred < 1:
        min_pos = int(np.ceil(len(y_true) * float(min_pos_pred)))
        min_pos = max(min_pos, 1)
    else:
        min_pos = int(min_pos_pred)

    thresholds = np.arange(tmin, tmax + 1e-12, step)

    best_t = 0.5
    best_s = -np.inf
    candidates = 0

    for t in thresholds:
        y_pred = (p_pos >= t).astype(int)
        n_pos_pred = int(y_pred.sum())
        if n_pos_pred < min_pos:
            continue

        candidates += 1

        if optimize_for == "f1":
            s = float(f1_score(y_true, y_pred, zero_division=0))
        elif optimize_for == "accuracy":
            s = float(accuracy_score(y_true, y_pred))
        else:  # bac
            s = float(balanced_accuracy_score(y_true, y_pred))

        if s > best_s:
            best_s = s
            best_t = float(t)

    # fallback: se nenhum candidato passou, usa quantil para garantir min_pos
    fallback = False
    if candidates == 0:
        fallback = True
        sorted_p = np.sort(p_pos)
        # quer que min_pos sejam >= t  => t = p no quantil (1 - min_pos/n)
        q = 1.0 - (min_pos / max(len(sorted_p), 1))
        q = float(np.clip(q, 0.0, 1.0))
        best_t = float(np.quantile(sorted_p, q))
        # score calculado no fallback
        y_pred = (p_pos >= best_t).astype(int)
        if optimize_for == "f1":
            best_s = float(f1_score(y_true, y_pred, zero_division=0))
        elif optimize_for == "accuracy":
            best_s = float(accuracy_score(y_true, y_pred))
        else:
            best_s = float(balanced_accuracy_score(y_true, y_pred))

    return best_t, best_s, candidates, fallback, min_pos

def _debug_prob(name, p):
    p = np.asarray(p)
    qs = np.quantile(p, [0.0, 0.1, 0.5, 0.9, 1.0])
    print(f"\n[DEBUG] Probabilidades {name}:")
    print(f"min={qs[0]:.6f} | p10={qs[1]:.6f} | p50={qs[2]:.6f} | p90={qs[3]:.6f} | max={qs[4]:.6f}")

# ===================== (1) Inner CV (com threshold=0.5 apenas para log) =====================
inner_fold_rows = []
for fold_id, (tr_idx, va_idx) in enumerate(cv_for_optuna.split(X_train8, y_train8), start=1):
    X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
    y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

    model_inner = XGBClassifier(**xgb_params)
    model_inner.fit(X_tr, y_tr)

    p_va = model_inner.predict_proba(X_va)[:, 1]
    metrics = all_metrics_table_long(y_va, p_va, threshold=0.5, eps=EPS)
    inner_fold_rows.append({"etapa": "inner_cv", "fold": fold_id, **metrics})

df_inner = pd.DataFrame(inner_fold_rows)

# ===================== (2) Validação hold-out: treina em train8 e escolhe threshold =====================
model_val = XGBClassifier(**xgb_params)
model_val.fit(X_train8, y_train8)
p_val = model_val.predict_proba(X_val1)[:, 1]

_debug_prob("na validação hold-out", p_val)

best_thr, best_thr_score, n_candidates, fallback, min_pos = find_best_threshold_robust(
    y_true=y_val1, p_pos=p_val,
    optimize_for=THRESH_OPTIMIZE_FOR,
    tmin=THRESH_MIN, tmax=THRESH_MAX, step=THRESH_GRID_STEP,
    min_pos_pred=MIN_POS_PRED
)
print(f"\n🎚️ Threshold escolhido ROBUSTO: {best_thr:.3f} | score({THRESH_OPTIMIZE_FOR})={best_thr_score:.6f} "
      f"| candidatos válidos={n_candidates} | fallback={fallback} | min_pos={min_pos}")

val_metrics_long = all_metrics_table_long(y_val1, p_val, threshold=best_thr, eps=EPS)
df_val = pd.DataFrame([{"etapa": "validacao", "fold": 9, **val_metrics_long}])

# ===================== (3) Treino final (8+1) e teste hold-out (threshold fixo) =====================
X_train9 = pd.concat([X_train8, X_val1], axis=0)
y_train9 = pd.concat([y_train8, y_val1], axis=0)

xgb_final = XGBClassifier(**xgb_params)
xgb_final.fit(X_train9, y_train9)
p_test = xgb_final.predict_proba(X_test1)[:, 1]

_debug_prob("no teste hold-out", p_test)

test_metrics_long = all_metrics_table_long(y_test1, p_test, threshold=best_thr, eps=EPS)
df_test = pd.DataFrame([{"etapa": "teste", "fold": 10, **test_metrics_long}])

# ===================== Consolida =====================
df_all = pd.concat([df_inner, df_val, df_test], ignore_index=True)

print("\n=== Resultados XGBoost 8/1/1 (inner, validação e teste) ===")
print(df_all.to_string(index=False))

# ===================== Tabela final (val/test) no formato da dissertação =====================
val_tab  = metrics_for_table(y_val1,  p_val,  threshold=best_thr, eps=EPS)
test_tab = metrics_for_table(y_test1, p_test, threshold=best_thr, eps=EPS)

df_final = pd.DataFrame([
    {"split": "validacao", "threshold": best_thr, **val_tab},
    {"split": "teste",     "threshold": best_thr, **test_tab},
])

cols_order = ["split","threshold","ACC","F1","BAC","PRE","REC","AUC","CE-C0","CE-C1","CE-C2","y","TP%","FP%","TN%","FN%"]
df_final = df_final[cols_order]

print("\n=== MÉTRICAS FINAIS (8/1/1) — threshold selecionado na validação hold-out ===")
print(df_final.to_string(index=False))

# ===================== Salva CSVs =====================
if CSV_OUT is not None:
    CSV_OUT = Path(CSV_OUT)
    CSV_OUT.mkdir(parents=True, exist_ok=True)

    df_inner.to_csv(CSV_OUT / "cv_811_inner_folds.csv", index=False)
    df_val.to_csv(CSV_OUT / "cv_811_validation.csv", index=False)
    df_test.to_csv(CSV_OUT / "cv_811_test.csv", index=False)
    df_all.to_csv(CSV_OUT / "cv_811_results_long.csv", index=False)

    df_final_path = CSV_OUT / f"xgb_811_metrics_val_test_opt_{OPTIMIZE_METRIC}_thr_{THRESH_OPTIMIZE_FOR}_robust.csv"
    df_final.to_csv(df_final_path, index=False)

    thr_path = CSV_OUT / f"best_threshold_cv811_{THRESH_OPTIMIZE_FOR}_robust.json"
    with open(thr_path, "w") as f:
        json.dump({
            "threshold": best_thr,
            "optimize_for": THRESH_OPTIMIZE_FOR,
            "score": best_thr_score,
            "min_pos_pred": MIN_POS_PRED,
            "min_pos_count": int(min_pos),
            "range": [THRESH_MIN, THRESH_MAX],
            "step": THRESH_GRID_STEP,
            "fallback": bool(fallback),
            "candidates": int(n_candidates)
        }, f, indent=2)

    print(f"\n✅ CSV final salvo em: {df_final_path}")
    print(f"✅ Threshold salvo em: {thr_path}")
    print("✅ CSVs 8/1/1 salvos em:", CSV_OUT)
else:
    print("\nCSV_OUT não definido — CSVs não foram salvos (apenas print).")

# ===================== Salva modelo final =====================
if MODEL_DIR is not None:
    MODEL_DIR = Path(MODEL_DIR)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model_path = MODEL_DIR / f"xgb_model_cv811_final_opt_{OPTIMIZE_METRIC}.pkl"
    joblib.dump(xgb_final, model_path)
    print("✅ Modelo XGBoost final salvo em:", model_path)

# ===================== Salva estudo + params =====================
if CSV_OUT is not None:
    joblib.dump(study, CSV_OUT / f"optuna_xgb_cv811_{OPTIMIZE_METRIC}_{N_TRIALS_OPTUNA}trials.pkl")
    with open(CSV_OUT / f"best_xgb_params_cv811_{OPTIMIZE_METRIC}.json", "w") as f:
        json.dump(xgb_params, f, indent=2)


🎯 Optuna otimizando por: AUC
🎚️ Threshold robusto otimiza: F1 (validação hold-out)
🔒 Threshold range: [0.05, 0.95] | step=0.00
🧷 MIN_POS_PRED: 0.01 (fração ou inteiro)
✅ Dados: X=(94446, 26) | y=(94446,) | classes=[0 1] -> (0/1)
✅ pos=32898 | neg=61548 | scale_pos_weight=1.871


[I 2026-01-26 06:53:37,211] A new study created in memory with name: xgb_tpe_cv811_auc



=== Split 8/1/1 (hold-out) — CORRETO ===
Treino (8/10): 75558 | Validação hold-out (1/10): 9444 | Teste hold-out (1/10): 9444
✅ Validação e Teste NÃO entram no Optuna.



Best trial: 0. Best value: 0.787315:  20%|██        | 1/5 [03:37<14:31, 218.00s/it]

[I 2026-01-26 06:57:15,211] Trial 0 finished with value: 0.7873147355771879 and parameters: {'n_estimators': 550, 'max_depth': 12, 'learning_rate': 0.038052091892039265, 'subsample': 0.779597545259111, 'colsample_bytree': 0.5624074561769746, 'min_child_weight': 4, 'gamma': 0.4646688973455957, 'reg_alpha': 0.29154431891537513, 'reg_lambda': 0.2537815508265665, 'max_delta_step': 4}. Best is trial 0 with value: 0.7873147355771879.


...

[I 2026-01-26 07:00:32,865] Trial 2 finished with value: 0.7812420254862049 and parameters: {'n_estimators': 645, 'max_depth': 6, 'learning_rate': 0.011239505740409895, 'subsample': 0.7099085529881075, 'colsample_bytree': 0.6824279936868144, 'min_child_weight': 9, 'gamma': 1.5973902572668779, 'reg_alpha': 0.011400863701127324, 'reg_lambda': 0.23423849847112907, 'max_delta_step': 0}. Best is trial 0 with value: 0.7873147355771879.


Best trial: 0. Best value: 0.787315:  80%|████████  | 4/5 [08:49<01:59, 119.67s/it]

[I 2026-01-26 07:02:26,804] Trial 3 finished with value: 0.7746224686137099 and parameters: {'n_estimators': 643, 'max_depth': 7, 'learning_rate': 0.0059882500578222475, 'subsample': 0.884665661176, 'colsample_bytree': 0.8862528132298237, 'min_child_weight': 9, 'gamma': 2.4369101533869655, 'reg_alpha': 0.00024586032763280086, 'reg_lambda': 0.5456725485601477, 'max_delta_step': 3}. Best is trial 0 with value: 0.7873147355771879.


Best trial: 0. Best value: 0.787315: 100%|██████████| 5/5 [10:52<00:00, 130.57s/it]


[I 2026-01-26 07:04:30,051] Trial 4 finished with value: 0.7726324646019287 and parameters: {'n_estimators': 448, 'max_depth': 9, 'learning_rate': 0.005500192756455945, 'subsample': 0.8727961206236347, 'colsample_bytree': 0.6035119926400068, 'min_child_weight': 8, 'gamma': 2.4936886087152876, 'reg_alpha': 0.012030178871154668, 'reg_lambda': 0.1537592023548176, 'max_delta_step': 1}. Best is trial 0 with value: 0.7873147355771879.

🏆 Melhor score (AUC, CV interna 8 folds): 0.787315
Best trial: 0

=== xgb_params finais (8/1/1) ===
n_estimators = 550
max_depth = 12
learning_rate = 0.038052091892039265
subsample = 0.779597545259111
colsample_bytree = 0.5624074561769746
min_child_weight = 4
gamma = 0.4646688973455957
reg_alpha = 0.29154431891537513
reg_lambda = 0.2537815508265665
max_delta_step = 4
tree_method = hist
n_jobs = 8
random_state = 42
verbosity = 0
objective = binary:logistic
eval_metric = auc
scale_pos_weight = 1.8708736093379537

[DEBUG] Probabilidades na validação hold-out:
min

In [ ]:
# ============================
# XGBoost 8/1/1 com Optuna (TPE + Pruning) + métricas e CSVs — alinhado ao RF SEM TRESHOLD  XXXX
# ============================
import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, log_loss, confusion_matrix
)
import time, joblib
from pathlib import Path

# --------- Monta X / y ---------
y = df['y'].copy()
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower and c != 'y']
FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]

X_model = df[FEATURES].copy()
X_model.replace([np.inf, -np.inf], np.nan, inplace=True)
X_model = X_model.fillna(X_model.median(numeric_only=True))

classes = np.unique(y.dropna().values)
n_classes = len(classes)
is_binary = (n_classes == 2)

if is_binary:
    c0, c1 = classes[0], classes[1]
    neg = int((y == c0).sum())
    pos = int((y == c1).sum())
    scale_pos_weight_default = float(neg / max(pos, 1))
else:
    scale_pos_weight_default = 1.0

# --------- Configs ---------
N_TRIALS_OPTUNA   = 5  # reduzido de 60 para 30 (otimização mais rápida)
EARLY_STOP_ROUNDS = 100
RANDOM_STATE      = 42
N_JOBS            = 8

# threshold global (se não existir)
try:
    THRESHOLD
except NameError:
    THRESHOLD = 0.5

EPS = 1e-12

# ===================== 8/1/1: define folds externos (10) =====================
outer_kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(outer_kf.split(X_model, y))

# Último fold = teste, penúltimo = validação, demais 8 = treino
test1_idx = fold_indices[-1][1]   # fold 10
val1_idx  = fold_indices[-2][1]   # fold 9
# Conjunto de treino = todos os índices que NÃO estão em val/test
train8_idx = np.setdiff1d(np.arange(len(X_model)), np.concatenate([val1_idx, test1_idx]))

X_train8, y_train8 = X_model.iloc[train8_idx], y.iloc[train8_idx]
X_val1,   y_val1   = X_model.iloc[val1_idx],   y.iloc[val1_idx]
X_test1,  y_test1  = X_model.iloc[test1_idx],  y.iloc[test1_idx]

# ===================== Optuna (CV interna = 8 folds no conjunto de treino) =====================
cv_for_optuna = StratifiedKFold(n_splits=8, shuffle=True, random_state=RANDOM_STATE)

def objective(trial: optuna.Trial) -> float:
    params = {
        # Mantém muitos estimadores, mas evita valores muito baixos
        "n_estimators": trial.suggest_int("n_estimators", 400, 800),
        
        # Seu ótimo foi 12 → ainda permite árvores fundas, mas corta profundidades muito rasas
        "max_depth": trial.suggest_int("max_depth", 6, 12),
        
        # Ótimo ≈ 0.016 → foca em taxas baixas, típicas de modelos mais estáveis
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.08, log=True),
        
        # Ótimo ≈ 0.72 → evita extremos de 0.5 e 1.0, concentra na região mais provável
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        
        # Ótimo ≈ 0.74 → mesma ideia: restringe para 0.5–0.9
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),
        
        # Ótimo = 7 → evita valores 1–2 muito frouxos e 12 muito duros
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 10),
        
        # Ótimo ≈ 5.5, mas ainda deixa gama pequena possível
        "gamma": trial.suggest_float("gamma", 0.0, 8.0),
        
        # Regularização L1: mais concentrada numa faixa razoável
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
        
        # Regularização L2: força a sair do “quase zero”
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        
        # Mantém range pequeno; seu ótimo foi 5
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 6),

        "tree_method": "hist",
        "n_jobs":      N_JOBS,
        "random_state": RANDOM_STATE,
        "verbosity":   0,
    }


    if is_binary:
        params.update({
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "scale_pos_weight": scale_pos_weight_default,
        })
        maximize_metric = True
    else:
        params.update({
            "objective": "multi:softprob",
            "num_class": n_classes,
            "eval_metric": "mlogloss",
        })
        maximize_metric = False

    fold_scores = []
    for fold_id, (tr_idx, va_idx) in enumerate(cv_for_optuna.split(X_train8, y_train8), 1):
        X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
        y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

        model = XGBClassifier(**params)

        use_callbacks = hasattr(xgb, "callback") and hasattr(xgb.callback, "EarlyStopping")
        if use_callbacks:
            es_cb = xgb.callback.EarlyStopping(
                rounds=EARLY_STOP_ROUNDS,
                save_best=True,
                maximize=maximize_metric
            )
            try:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[es_cb])
            except TypeError:
                model.fit(X_tr, y_tr)
        else:
            try:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
            except TypeError:
                model.fit(X_tr, y_tr)

        proba_va = model.predict_proba(X_va)
        if is_binary:
            score = roc_auc_score(y_va, proba_va[:, 1])
        else:
            score = roc_auc_score(y_va, proba_va, multi_class="ovr", average="macro")

        fold_scores.append(float(score))
        trial.report(float(np.mean(fold_scores)), fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))

sampler = TPESampler(seed=RANDOM_STATE, multivariate=True, group=True)
pruner  = MedianPruner(n_warmup_steps=2)

study = optuna.create_study(
    study_name="xgb_tpe_auc_8fold_search",
    direction="maximize",
    sampler=sampler,
    pruner=pruner
)
study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)

print("Best value (AUROC, CV interna 8 folds):", study.best_value)
print("Best trial:", study.best_trial.number)

best = study.best_trial.params.copy()

# --------- Dicionário final para reuso ---------
xgb_params = {
    "n_estimators":      int(best.get("n_estimators", 300)),
    "max_depth":         int(best.get("max_depth", 6)),
    "learning_rate":     float(best.get("learning_rate", 0.05)),
    "subsample":         float(best.get("subsample", 0.8)),
    "colsample_bytree":  float(best.get("colsample_bytree", 0.8)),
    "min_child_weight":  int(best.get("min_child_weight", 1)),
    "gamma":             float(best.get("gamma", 0.0)),
    "reg_alpha":         float(best.get("reg_alpha", 0.0)),
    "reg_lambda":        float(best.get("reg_lambda", 1.0)),
    "max_delta_step":    int(best.get("max_delta_step", 0)),
    "tree_method":       "hist",
    "n_jobs":            N_JOBS,
    "random_state":      RANDOM_STATE,
    "verbosity":         0,
}
if is_binary:
    xgb_params.update({
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "scale_pos_weight": scale_pos_weight_default
    })
else:
    xgb_params.update({
        "objective": "multi:softprob",
        "eval_metric": "mlogloss",
        "num_class": n_classes
    })

print("\n=== xgb_params finais (8/1/1 corrigido) ===")
for k, v in xgb_params.items():
    print(f"{k} = {v}")

# ===================== Métricas no mesmo formato do Random Forest =====================

def cross_entropy_per_class(y_true, p_pos, eps=1e-12):
    """
    Versão idêntica à usada no RF.
    y_true: [0/1]; p_pos: probabilidade prevista da classe 1.
    Retorna: CE0 (média de -log(1-p) em y=0) e CE1 (média de -log(p) em y=1).
    """
    y_true = np.asarray(y_true).astype(int)
    p_pos = np.clip(np.asarray(p_pos), eps, 1 - eps)
    mask0 = (y_true == 0)
    mask1 = ~mask0
    ce0 = float(np.mean(-np.log(1 - p_pos[mask0]))) if np.any(mask0) else np.nan
    ce1 = float(np.mean(-np.log(p_pos[mask1])))     if np.any(mask1) else np.nan
    return ce0, ce1

def all_metrics_table(y_true, proba_pos, threshold=0.5, eps=1e-12):
    """
    Copiado do RF: retorna dict com TODAS as métricas:
    - Acurácia, Cross-Entropy C0/C1, Proporção C0/C1, F1, Balanced Accuracy,
      Precision, Recall, TP/FP/TN/FN (abs) e TP/FP/TN/FN (% do total).
    """
    y_true = np.asarray(y_true).astype(int)
    proba_pos = np.asarray(proba_pos)
    y_pred = (proba_pos >= threshold).astype(int)

    # Contagens
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN

    # Proporções por classe (no y_true)
    n0 = int(np.sum(y_true == 0))
    n1 = int(np.sum(y_true == 1))
    prop_c0 = n0 / total if total > 0 else np.nan
    prop_c1 = n1 / total if total > 0 else np.nan

    # Cross-Entropy por classe
    ce0, ce1 = cross_entropy_per_class(y_true, proba_pos, eps=eps)

    # Métricas padrão
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)

    # Balanced Accuracy = (recall_0 + recall_1)/2
    recall_1 = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    recall_0 = TN / (TN + FP) if (TN + FP) > 0 else np.nan
    bal_acc  = np.nanmean([recall_0, recall_1])

    # Percentuais de confusão
    TP_pct = TP / total if total > 0 else np.nan
    FP_pct = FP / total if total > 0 else np.nan
    TN_pct = TN / total if total > 0 else np.nan
    FN_pct = FN / total if total > 0 else np.nan

    return {
        "Acurácia": acc,
        "Cross-Entropy C0": ce0,
        "Proporção C0": prop_c0,
        "Cross-Entropy C1": ce1,
        "Proporção C1": prop_c1,
        "F1 Score": f1,
        "Balanced Accuracy": bal_acc,
        "Precision": prec,
        "Recall": rec,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "TP(%)": TP_pct, "FP(%)": FP_pct, "TN(%)": TN_pct, "FN(%)": FN_pct,
    }

# ===================== (1) Reavalia CV interna (8 folds) com melhores params =====================
inner_fold_rows = []

for fold_id, (tr_idx, va_idx) in enumerate(cv_for_optuna.split(X_train8, y_train8), start=1):
    X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
    y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

    model_inner = XGBClassifier(**xgb_params)
    model_inner.fit(X_tr, y_tr)

    # Apenas binário: usa prob da classe positiva
    proba_va = model_inner.predict_proba(X_va)[:, 1]
    metrics = all_metrics_table(y_va, proba_va, threshold=THRESHOLD, eps=EPS)
    row = {"etapa": "inner_cv", "fold": fold_id, **metrics}
    inner_fold_rows.append(row)

df_inner = pd.DataFrame(inner_fold_rows)

# ===================== (2) Validação intermediária (fold 9) =====================
t_val_ini = time.time()

model_val = XGBClassifier(**xgb_params)
model_val.fit(X_train8, y_train8)
proba_val = model_val.predict_proba(X_val1)[:, 1]

val_metrics_dict = all_metrics_table(y_val1, proba_val, threshold=THRESHOLD, eps=EPS)
val_row = {"etapa": "validacao", "fold": 9, **val_metrics_dict}
df_val = pd.DataFrame([val_row])

# ===================== (3) Treino final (8+1 = 9 folds) e teste (fold 10) =====================
X_train9 = pd.concat([X_train8, X_val1])
y_train9 = pd.concat([y_train8, y_val1])

t_test_ini = time.time()

xgb_final = XGBClassifier(**xgb_params)
xgb_final.fit(X_train9, y_train9)
proba_test = xgb_final.predict_proba(X_test1)[:, 1]

test_metrics_dict = all_metrics_table(y_test1, proba_test, threshold=THRESHOLD, eps=EPS)
test_row = {"etapa": "teste", "fold": 10, **test_metrics_dict}
df_test = pd.DataFrame([test_row])

# ===================== Consolida e salva CSVs iguais ao RF =====================
df_all = pd.concat([df_inner, df_val, df_test], ignore_index=True)

# Garante que CSV_OUT exista
try:
    CSV_OUT
except NameError:
    CSV_OUT = None

if CSV_OUT is not None:
    CSV_OUT = Path(CSV_OUT)
    CSV_OUT.mkdir(parents=True, exist_ok=True)

    # mesmos nomes do RF (assumindo que CSV_OUT é diferente para RF e XGB)
    df_inner.to_csv(CSV_OUT / "cv_811_inner_folds.csv", index=False)
    df_val.to_csv(CSV_OUT / "cv_811_validation.csv", index=False)
    df_test.to_csv(CSV_OUT / "cv_811_test.csv", index=False)
    df_all.to_csv(CSV_OUT / "cv_811_results_long.csv", index=False)

    # também mantém um resumo compacto específico do XGB (val + teste)
    df_xgb_811_results = pd.concat([df_val, df_test], ignore_index=True)
    df_xgb_811_results.to_csv(CSV_OUT / "xgb_cv_811_results.csv", index=False)

    print("\nCSVs XGBoost 8/1/1 salvos em:", CSV_OUT)
else:
    print("\nCSV_OUT não definido — CSVs não foram salvos.")

print("\n=== Resultados XGBoost 8/1/1 (inner, validação e teste) ===")
print(df_all.to_string(index=False))

# ===================== Salva modelo final (compatível com seu fluxo) =====================
try:
    MODEL_DIR
except NameError:
    MODEL_DIR = None

if MODEL_DIR is not None:
    MODEL_DIR = Path(MODEL_DIR)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model_path = MODEL_DIR / "xgb_model_cf10_8_1_1_final.pkl"
    joblib.dump(xgb_final, model_path)
    print("Modelo XGBoost final salvo em:", model_path)


In [ ]:
# Preparação dos dados X, y
y = df['y']

# Detecta colunas identificadoras a partir de EXCLUDE_COLUMNS (case-insensitive)
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
# garante que id columns não contenham a coluna alvo y
ID_COLS = [c for c in ID_COLS if c != 'y']

# FEATURES são todas as colunas exceto ids e y
FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]

# X_model será usado para treinar (sem ids)
X_model = df[FEATURES].copy()
X_model.replace([np.inf, -np.inf], np.nan, inplace=True)
# preenche valores faltantes simples (pode ser alterado)
X_model = X_model.fillna(X_model.median(numeric_only=True))

from sklearn.metrics import roc_curve  # necessário pois é usado abaixo
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, log_loss, confusion_matrix, roc_auc_score, auc
)
from tqdm import tqdm
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import matplotlib.pyplot as plt

# ==== Seleção 9/1 consistente com o esquema 8/1/1 ====
# Se já existem test1_idx/val1_idx/train8_idx do bloco 8/1/1, usa o mesmo teste (fold 10)
if 'test1_idx' in globals():
    # treino 9/1 = todos os índices menos o teste
    all_idx = np.arange(len(X_model))
    train9_idx = np.setdiff1d(all_idx, test1_idx)
    _selected_splits = [(train9_idx, test1_idx)]
    # define o número de folds para o rótulo (ex.: 10)
    try:
        N_SPLITS
    except NameError:
        N_SPLITS = 10
else:
    # fallback: comportamento antigo (pega o último split do KFold)
    try:
        N_SPLITS
    except NameError:
        N_SPLITS = 10
    kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    _all_splits = list(kf.split(X_model, y))
    _last_train_idx, _last_test_idx = _all_splits[-1]
    _selected_splits = [(_last_train_idx, _last_test_idx)]

# estruturas de resultado
resultados = []
# listas para métricas agregadas por fold (treino/teste)
aurocs_train = []
aurocs_test = []
fprs_train = []
tprs_train = []
fprs_test = []
tprs_test = []
importancias_lista = []
X_train_parts = []
X_test_parts = []
y_train_parts = []
y_test_parts = []
y_train_pred_parts = []
y_test_pred_parts = []

start_total = time.time()

with PdfPages(PDF_OUT / f'XGB_CV_{N_SPLITS}.pdf') as pdf:
    # total=1 porque só executaremos o último split => 9/1
    with tqdm(total=1, desc="🔄 Folds K-Fold (9/1)") as pbar:
        # define o "fold" como N_SPLITS (ex.: 10) para marcar que é o último
        for fold, (train_idx, test_idx) in enumerate(_selected_splits, start=N_SPLITS):
            t0 = time.time()
            # Para manter ids nos arquivos de saída, indexamos sobre o DataFrame original
            X_train_full = df.iloc[train_idx][ID_COLS + FEATURES] if len(ID_COLS) > 0 else df.iloc[train_idx][FEATURES]
            X_test_full  = df.iloc[test_idx][ID_COLS + FEATURES]  if len(ID_COLS) > 0 else df.iloc[test_idx][FEATURES]
            # Usamos apenas as FEATURES (sem ids) para treinar o modelo
            X_train = X_train_full[FEATURES].copy()
            X_test  = X_test_full[FEATURES].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            model = XGBClassifier(**xgb_params)
            model.fit(X_train, y_train)

            # Previsões de probabilidade
            y_proba_test_all = model.predict_proba(X_test)
            y_proba_test_C1 = y_proba_test_all[:, 1]
            y_proba_test_C1_bin = (y_proba_test_C1 >= THRESHOLD).astype(int)

            y_proba_train_all = model.predict_proba(X_train)
            y_proba_train_C1 = y_proba_train_all[:, 1]
            y_proba_train_C1_bin = (y_proba_train_C1 >= THRESHOLD).astype(int)

            # guarda partes completas (incluem ids) para concat posterior e rastreabilidade
            # mantemos o índice original para fácil merge posterior
            X_test_parts.append(X_test_full.assign(fold=fold).reset_index())
            y_test_parts.append(pd.Series(y_test, name='y_test').to_frame().assign(fold=fold).reset_index())
            y_test_pred_parts.append(
                pd.DataFrame(y_proba_test_all, columns=['prob_0', 'prob_1'], index=X_test_full.index)
                  .assign(fold=fold).reset_index()
            )

            X_train_parts.append(X_train_full.assign(fold=fold).reset_index())
            y_train_parts.append(pd.Series(y_train, name='y_train').to_frame().assign(fold=fold).reset_index())
            y_train_pred_parts.append(
                pd.DataFrame(y_proba_train_all, columns=['prob_0', 'prob_1'], index=X_train_full.index)
                  .assign(fold=fold).reset_index()
            )

            # Métricas Treino
            cm_train = confusion_matrix(y_train, y_proba_train_C1_bin)
            tn_tr, fp_tr, fn_tr, tp_tr = cm_train.ravel()
            total_cm_treino = cm_train.sum()
            fpr_tr, tpr_tr, _ = roc_curve(y_train, y_proba_train_C1)
            fprs_train.append(fpr_tr)
            tprs_train.append(tpr_tr)
            auroc_tr_val = roc_auc_score(y_train, y_proba_train_C1)
            aurocs_train.append(auroc_tr_val)

            MASK_0_TRAIN = (y_train == 0)
            MASK_1_TRAIN = (y_train == 1)
            logloss_train_0 = log_loss(
                y_train[MASK_0_TRAIN], y_proba_train_C1[MASK_0_TRAIN], labels=[0, 1]
            ) if MASK_0_TRAIN.sum() > 0 else np.nan
            logloss_train_1 = log_loss(
                y_train[MASK_1_TRAIN], y_proba_train_C1[MASK_1_TRAIN], labels=[0, 1]
            ) if MASK_1_TRAIN.sum() > 0 else np.nan
            cross_prop_train = pd.Series(y_train).value_counts(normalize=True) * 100

            # Métricas Teste
            cm_test = confusion_matrix(y_test, y_proba_test_C1_bin)
            tn_ts, fp_ts, fn_ts, tp_ts = cm_test.ravel()
            total_cm_teste = cm_test.sum()
            fpr_ts, tpr_ts, _ = roc_curve(y_test, y_proba_test_C1)
            fprs_test.append(fpr_ts)
            tprs_test.append(tpr_ts)
            auroc_ts_val = roc_auc_score(y_test, y_proba_test_C1)
            aurocs_test.append(auroc_ts_val)

            MASK_0_TEST = (y_test == 0)
            MASK_1_TEST = (y_test == 1)
            logloss_test_0 = log_loss(
                y_test[MASK_0_TEST], y_proba_test_C1[MASK_0_TEST], labels=[0, 1]
            ) if MASK_0_TEST.sum() > 0 else np.nan
            logloss_test_1 = log_loss(
                y_test[MASK_1_TEST], y_proba_test_C1[MASK_1_TEST], labels=[0, 1]
            ) if MASK_1_TEST.sum() > 0 else np.nan
            cross_prop_test = pd.Series(y_test).value_counts(normalize=True) * 100

            # Armazena resultados (Treino/Teste)
            resultados.append({
                'Conjunto': 'Treino(9folds)',
                'Fold': fold,
                'Acurácia': accuracy_score(y_train, y_proba_train_C1_bin),
                'Cross-Entropy C0': logloss_train_0,
                'Proporção C0': cross_prop_train.iloc[0] if 0 in cross_prop_train.index else np.nan,
                'Cross-Entropy C1': logloss_train_1,
                'Proporção C1': cross_prop_train.iloc[1] if 1 in cross_prop_train.index else np.nan,
                'F1 Score': f1_score(y_train, y_proba_train_C1_bin),
                'Balanced Accuracy': balanced_accuracy_score(y_train, y_proba_train_C1_bin),
                'Precision': precision_score(y_train, y_proba_train_C1_bin),
                'Recall': recall_score(y_train, y_proba_train_C1_bin),
                'AUROC': auroc_tr_val,
                'TP': tp_tr, 'FP': fp_tr, 'TN': tn_tr, 'FN': fn_tr,
                'TP(%)': round(tp_tr / total_cm_treino * 100, 2),
                'FP(%)': round(fp_tr / total_cm_treino * 100, 2),
                'TN(%)': round(tn_tr / total_cm_treino * 100, 2),
                'FN(%)': round(fn_tr / total_cm_treino * 100, 2),
                'Tempo (s)': round(time.time() - t0, 2)
            })

            resultados.append({
                'Conjunto': 'Teste(fold10)',
                'Fold': fold,
                'Acurácia': accuracy_score(y_test, y_proba_test_C1_bin),
                'Cross-Entropy C0': logloss_test_0,
                'Proporção C0': cross_prop_test.iloc[0] if 0 in cross_prop_test.index else np.nan,
                'Cross-Entropy C1': logloss_test_1,
                'Proporção C1': cross_prop_test.iloc[1] if 1 in cross_prop_test.index else np.nan,
                'F1 Score': f1_score(y_test, y_proba_test_C1_bin),
                'Balanced Accuracy': balanced_accuracy_score(y_test, y_proba_test_C1_bin),
                'Precision': precision_score(y_test, y_proba_test_C1_bin),
                'Recall': recall_score(y_test, y_proba_test_C1_bin),
                'AUROC': auroc_ts_val,
                'TP': tp_ts, 'FP': fp_ts, 'TN': tn_ts, 'FN': fn_ts,
                'TP(%)': round(tp_ts / total_cm_teste * 100, 2),
                'FP(%)': round(fp_ts / total_cm_teste * 100, 2),
                'TN(%)': round(tn_ts / total_cm_teste * 100, 2),
                'FN(%)': round(fn_ts / total_cm_teste * 100, 2),
                'Tempo (s)': round(time.time() - t0, 2)
            })

            # Importância das features (percentual)
            importancias = model.feature_importances_
            nomes_colunas = FEATURES
            linha_importancia = {'Conjunto': 'Teste(fold10)', 'Fold': fold}
            for nome, valor in zip(nomes_colunas, importancias):
                linha_importancia[nome] = f"{valor * 100:.2f}%"
            importancias_lista.append(linha_importancia)

            # salva modelo por fold (aqui só 1, 9/1)
            model_path = MODEL_DIR / f'xgb_model_cf{N_SPLITS}_9_1.pkl'
            joblib.dump(model, model_path)

            pbar.update(1)

    # fim do "loop" (apenas 1 split 9/1)

# Concatena e salva os resultados
df_resultados = pd.DataFrame(resultados)
# adiciona média das métricas (aqui será idêntica, pois só há 1 split, mas mantém compatibilidade)
mean_row = df_resultados.select_dtypes(include=[np.number]).mean()
mean_row['Conjunto'] = 'Média'
mean_row['Fold'] = 'Média'
df_resultados = pd.concat([df_resultados, pd.DataFrame([mean_row])], ignore_index=True, sort=False)

# salva csvs principais
df_resultados.to_csv(CSV_OUT / f'xgb_cv_results_{N_SPLITS}_9_1.csv', index=False)
pd.DataFrame(importancias_lista).to_csv(CSV_OUT / f'xgb_feature_importances_{N_SPLITS}_9_1.csv', index=False)

# Concatena blocos para gerar dataframes completos de treino/teste com probabilidades
if len(X_train_parts) > 0:
    X_train_global = pd.concat(X_train_parts, ignore_index=True)
    y_train_global = pd.concat(y_train_parts, ignore_index=True)
    y_train_pred_global = pd.concat(y_train_pred_parts, ignore_index=True)
else:
    X_train_global = pd.DataFrame(); y_train_global = pd.DataFrame(); y_train_pred_global = pd.DataFrame()

if len(X_test_parts) > 0:
    X_test_global = pd.concat(X_test_parts, ignore_index=True)
    y_test_global = pd.concat(y_test_parts, ignore_index=True)
    y_test_pred_global = pd.concat(y_test_pred_parts, ignore_index=True)
else:
    X_test_global = pd.DataFrame(); y_test_global = pd.DataFrame(); y_test_pred_global = pd.DataFrame()

# Ajusta nomes: os objetos concat possuem coluna 'index' que era o índice original; renomeamos para 'orig_index' para merge limpo
for df_block in [X_train_global, X_test_global, y_train_global, y_test_global, y_train_pred_global, y_test_pred_global]:
    if 'index' in df_block.columns:
        df_block.rename(columns={'index': 'orig_index'}, inplace=True)

# Mescla para criar arquivos finais com probas e fold
if not X_train_global.empty and not y_train_global.empty:
    train_df = X_train_global.merge(y_train_global, on=['orig_index', 'fold'], how='left')
    if not y_train_pred_global.empty:
        train_df = train_df.merge(y_train_pred_global, on=['orig_index', 'fold'], how='left')
else:
    train_df = pd.DataFrame()

if not X_test_global.empty and not y_test_global.empty:
    test_df = X_test_global.merge(y_test_global, on=['orig_index', 'fold'], how='left')
    if not y_test_pred_global.empty:
        test_df = test_df.merge(y_test_pred_global, on=['orig_index', 'fold'], how='left')
else:
    test_df = pd.DataFrame()

# Salva os datasets com probabilidades (prob_0, prob_1) e já binarizados por threshold
if not test_df.empty:
    test_df['y_proba'] = test_df['prob_1']
    test_df['y_pred'] = (test_df['y_proba'] >= THRESHOLD).astype(int)
    test_df.to_csv(CSV_OUT / f'xgb_test_with_proba_{N_SPLITS}_9_1.csv', index=False)
if not train_df.empty:
    train_df['y_proba'] = train_df['prob_1']
    train_df['y_pred'] = (train_df['y_proba'] >= THRESHOLD).astype(int)
    train_df.to_csv(CSV_OUT / f'xgb_train_with_proba_{N_SPLITS}_9_1.csv', index=False)

# Salva um dataset unificado em process (com y_proba e y_pred sobre todo o conjunto de teste concatenado)
if not test_df.empty:
    test_df.to_csv(Path('../data/processed') / f'wdbc_xgb_test_with_proba_{N_SPLITS}_9_1.csv', index=False)

# create a full-augmented dataset for the original input dataset with y, y_pred, y_proba
try:
    if not test_df.empty:
        df_aug = df.reset_index().rename(columns={'index': 'orig_index'})
        preds = test_df[['orig_index', 'y_pred', 'y_proba']]
        df_augmented = pd.merge(df_aug, preds, how='left', on='orig_index')
        df_augmented.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_preds_xgb_{N_SPLITS}_9_1.csv', index=False)
    else:
        print('test_df vazio: não foi possível gerar dataset augmentado com previsões')
except Exception as e_aug:
    print('Erro ao gerar dataset augmentado com previsões:', e_aug)

print('Resultados salvos em:', CSV_OUT)


In [ ]:
# # Diagnóstico rápido para NaN / tipos em train_df e test_df
# for name, ycol in [('train_df', 'y_train'), ('test_df', 'y_test')]:
#     df_block = globals().get(name)
#     if df_block is None:
#         print(f"{name} não existe")
#         continue
#     if df_block.empty:
#         print(f"{name} existe mas está vazio")
#         continue
#     print(f"\n{name}: shape={df_block.shape}")
#     for col in [ycol, 'prob_1', 'y_proba']:
#         if col in df_block.columns:
#             print(f"  {col}: nulls={df_block[col].isnull().sum()}, dtype={df_block[col].dtype}, unique_count={df_block[col].nunique(dropna=True)}")
#     cols_to_show = [c for c in [ycol, 'prob_1', 'y_proba'] if c in df_block.columns]
#     display(df_block[cols_to_show].head(10))

In [ ]:
# Geração de PDF resumo (com algumas páginas importantes)
with PdfPages(PDF_OUT / f'XGB_CV_SUMMARY_{N_SPLITS}.pdf') as pdf:
    # Página de parâmetros
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    parametros = xgb_params.copy()
    parametros['threshold']        = THRESHOLD
    # agora deixamos explícito o esquema 8/1/1
    parametros['outer_n_splits']   = N_SPLITS          # normalmente 10
    parametros['schema']           = '8/1/1 (treino: 8 folds, validação: 1 fold, teste: 1 fold)'
    parametros['inner_cv_folds']   = 8                 # CV interna do Optuna

    texto = 'Algoritmo: XGBoost\n' + '\n'.join([f'{k}: {v}' for k, v in parametros.items()])
    ax.text(0, 1, texto, fontsize=11, family='monospace', verticalalignment='top')
    ax.set_title('📌 Parâmetros do Modelo - XGBoost')
    # salva PNG dos parâmetros (opcional)
    try:
        png_path = IMAGES_OUT / f'params_{N_SPLITS}.png'
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
    except Exception:
        pass
    pdf.savefig(fig)
    plt.close()

    # Página de métricas resumidas
    if not df_resultados.empty:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.axis('off')
        display_df = (
            df_resultados
            .groupby(['Conjunto'])
            .mean(numeric_only=True)
            .T
            .round(4)
        )
        table = ax.table(
            cellText=display_df.values,
            colLabels=display_df.columns,
            rowLabels=display_df.index,
            loc='center'
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        try:
            png_path = IMAGES_OUT / f'metrics_summary_{N_SPLITS}.png'
            fig.savefig(png_path, dpi=300, bbox_inches='tight')
        except Exception:
            pass
        pdf.savefig(fig)
        plt.close()

    # Página ROC - Treino vs Teste (lado a lado)
    try:
        plotted = False
        # Tenta usar dataframes concatenados quando disponíveis (train_df/test_df)
        if 'y_train' in train_df.columns and 'prob_1' in train_df.columns:
            y_train_all = train_df['y_train']
            y_score_train_all = train_df['prob_1']
            fpr_train, tpr_train, _ = roc_curve(y_train_all, y_score_train_all)
            roc_auc_train = auc(fpr_train, tpr_train)
            plotted = True
        elif len(fprs_train) > 0:
            # média interpolada das curvas por fold (treino)
            mean_fpr = np.linspace(0, 1, 200)
            tprs = []
            for fpr, tpr in zip(fprs_train, tprs_train):
                tpr_i = np.interp(mean_fpr, fpr, tpr)
                tpr_i[0] = 0.0
                tprs.append(tpr_i)
            mean_tpr = np.mean(tprs, axis=0)
            mean_tpr[-1] = 1.0
            fpr_train, tpr_train = mean_fpr, mean_tpr
            roc_auc_train = np.mean(aurocs_train) if len(aurocs_train) > 0 else auc(fpr_train, tpr_train)
            plotted = True

        if 'y_test' in test_df.columns and 'prob_1' in test_df.columns:
            y_test_all = test_df['y_test']
            y_score_test_all = test_df['prob_1']
            fpr_test, tpr_test, _ = roc_curve(y_test_all, y_score_test_all)
            roc_auc_test = auc(fpr_test, tpr_test)
            plotted = True
        elif len(fprs_test) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = []
            for fpr, tpr in zip(fprs_test, tprs_test):
                tpr_i = np.interp(mean_fpr, fpr, tpr)
                tpr_i[0] = 0.0
                tprs.append(tpr_i)
            mean_tpr = np.mean(tprs, axis=0)
            mean_tpr[-1] = 1.0
            fpr_test, tpr_test = mean_fpr, mean_tpr
            roc_auc_test = np.mean(aurocs_test) if len(aurocs_test) > 0 else auc(fpr_test, tpr_test)
            plotted = True

        if plotted:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            # ROC Treino (esquerda)
            try:
                axes[0].plot(fpr_train, tpr_train, label=f'AUC = {roc_auc_train:.4f}')
                axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
                axes[0].set_title('ROC - Treino (8+1 folds)')
                axes[0].set_xlabel('FPR')
                axes[0].set_ylabel('TPR')
                axes[0].legend(loc='lower right')
            except Exception:
                axes[0].text(0.5, 0.5, 'Não há dados de treino para ROC', ha='center')
            # ROC Teste (direita)
            try:
                axes[1].plot(fpr_test, tpr_test, label=f'AUC = {roc_auc_test:.4f}')
                axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
                axes[1].set_title('ROC - Teste (fold 10)')
                axes[1].set_xlabel('FPR')
                axes[1].set_ylabel('TPR')
                axes[1].legend(loc='lower right')
            except Exception:
                axes[1].text(0.5, 0.5, 'Não há dados de teste para ROC', ha='center')
            plt.tight_layout()
            # salva PNG do ROC combinado
            try:
                png_path = IMAGES_OUT / f'roc_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
        else:
            print('Não foi possível gerar curvas ROC (dados ausentes)')
    except Exception as e:
        print('Não foi possível gerar ROC agregado:', e)

    # Página Confusion Matrix - Treino vs Teste (counts + percent)
    try:
        def build_cm_from_df(df_block, true_col, pred_col):
            # retorna cm e cm_percent
            y_true = df_block[true_col]
            y_pred = df_block[pred_col]
            cm = confusion_matrix(y_true, y_pred)
            total = cm.sum()
            cm_percent = cm / total * 100 if total > 0 else np.zeros_like(cm, dtype=float)
            return cm, cm_percent

        # tenta usar os dataframes concat (se disponíveis)
        cm_train, cm_train_pct = None, None
        cm_test, cm_test_pct = None, None
        if 'y_train' in train_df.columns and 'y_pred' in train_df.columns:
            cm_train, cm_train_pct = build_cm_from_df(train_df, 'y_train', 'y_pred')
        else:
            # fallback: agrega de df_resultados (soma de TP,FP,TN,FN)
            if 'TP' in df_resultados.columns:
                train_rows = df_resultados[df_resultados['Conjunto'] == 'Treino']
                TP = int(train_rows['TP'].sum()) if not train_rows.empty else 0
                FP = int(train_rows['FP'].sum()) if not train_rows.empty else 0
                TN = int(train_rows['TN'].sum()) if not train_rows.empty else 0
                FN = int(train_rows['FN'].sum()) if not train_rows.empty else 0
                cm_train = np.array([[TN, FP], [FN, TP]])
                total = cm_train.sum()
                cm_train_pct = cm_train / total * 100 if total > 0 else np.zeros_like(cm_train, dtype=float)

        if 'y_test' in test_df.columns and 'y_pred' in test_df.columns:
            cm_test, cm_test_pct = build_cm_from_df(test_df, 'y_test', 'y_pred')
        else:
            if 'TP' in df_resultados.columns:
                test_rows = df_resultados[df_resultados['Conjunto'] == 'Teste']
                TP = int(test_rows['TP'].sum()) if not test_rows.empty else 0
                FP = int(test_rows['FP'].sum()) if not test_rows.empty else 0
                TN = int(test_rows['TN'].sum()) if not test_rows.empty else 0
                FN = int(test_rows['FN'].sum()) if not test_rows.empty else 0
                cm_test = np.array([[TN, FP], [FN, TP]])
                total = cm_test.sum()
                cm_test_pct = cm_test / total * 100 if total > 0 else np.zeros_like(cm_test, dtype=float)

        # plota se ao menos uma matriz existir
        if (cm_train is None and cm_test is None):
            print('Não há dados suficientes para gerar matrizes de confusão.')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            def plot_cm(ax, cm, cm_pct, title):
                if cm is None:
                    ax.text(0.5, 0.5, 'Sem dados', ha='center', va='center')
                    ax.axis('off')
                    return
                annot = np.empty(cm.shape, dtype=object)
                for i in range(cm.shape[0]):
                    for j in range(cm.shape[1]):
                        annot[i, j] = f"{int(cm[i,j])}\n({cm_pct[i,j]:.2f}%)"
                sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=False, annot_kws={'size':12})
                ax.set_xlabel('Predito')
                ax.set_ylabel('Verdadeiro')
                ax.set_xticklabels(['0','1'])
                ax.set_yticklabels(['0','1'])
                ax.set_title(title)

            plot_cm(axes[0], cm_train, cm_train_pct, 'Matriz de Confusão - Treino (8+1 folds)')
            plot_cm(axes[1], cm_test,  cm_test_pct,  'Matriz de Confusão - Teste (fold 10)')

            plt.tight_layout()
            # salva PNG da matriz de confusão
            try:
                png_path = IMAGES_OUT / f'confusion_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
    except Exception as e:
        print('Não foi possível gerar matrizes de confusão:', e)

    # Página importâncias médias + Pareto + variâncias + violinos (inalterado)
    # (mantém exatamente o restante do seu bloco)


In [ ]:
# === XGBoost Basico (single train/test split) — usando o mesmo split 8/1/1 (train9 vs fold 10) ===
# Agora:
#   - Usa exatamente X_train9 / y_train9 como treino
#   - Usa exatamente X_test1  / y_test1  como teste
#   - TEST_SIZE é apenas alias para nomes de arquivos, calculado a partir desse split

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
import pandas as pd
import joblib
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

# ----------------- Verificações de pré-condição -----------------
# X_model e y montados no bloco 8/1/1
if 'X_model' not in globals() or 'y' not in globals():
    raise RuntimeError('X_model ou y não encontrados — execute o bloco 8/1/1 antes deste.')

# Splits gerados no bloco 8/1/1
if 'X_train9' not in globals() or 'X_test1' not in globals() or 'y_train9' not in globals() or 'y_test1' not in globals():
    raise RuntimeError('X_train9/X_test1/y_train9/y_test1 não encontrados — execute o bloco 8/1/1 antes deste.')

# Parâmetros auxiliares (apenas para nomes)
try:
    RT_RANDOM_STATE
except NameError:
    RT_RANDOM_STATE = 42

# TEST_SIZE agora é derivado do tamanho do teste em relação ao dataset total
TEST_SIZE = float(len(X_test1)) / float(len(X_model))
TEST_SIZE_PCT = int(round(TEST_SIZE * 100))

# ----------------- Define X, y e splits reaproveitando o 8/1/1 -----------------
# Mantém a semântica original: X, y, X_train, X_test, y_train, y_test
X = X_model.copy()
y_all = y.copy()

X_train = X_train9.copy()
X_test  = X_test1.copy()
y_train = y_train9.copy()
y_test  = y_test1.copy()

# ----------------- Treino do modelo básico com os melhores parâmetros xgb_params -----------------
if 'xgb_params' not in globals():
    raise RuntimeError('xgb_params não encontrado — execute o bloco de Optuna 8/1/1 antes deste.')

model_basic = XGBClassifier(**xgb_params)
model_basic.fit(X_train, y_train)

# Predições binárias
y_pred_train_bin = model_basic.predict(X_train)
y_pred_test_bin  = model_basic.predict(X_test)

# Probabilidades (para AUROC)
y_proba_train = model_basic.predict_proba(X_train)[:, 1]
y_proba_test  = model_basic.predict_proba(X_test)[:, 1]

# Métricas principais
acc_train = accuracy_score(y_train, y_pred_train_bin)
acc_test  = accuracy_score(y_test,  y_pred_test_bin)
f1_tr     = f1_score(y_train, y_pred_train_bin)
f1_ts     = f1_score(y_test,  y_pred_test_bin)
prec_tr   = precision_score(y_train, y_pred_train_bin)
prec_ts   = precision_score(y_test,  y_pred_test_bin)
rec_tr    = recall_score(y_train, y_pred_train_bin)
rec_ts    = recall_score(y_test,  y_pred_test_bin)

auc_tr = roc_auc_score(y_train, y_proba_train) if len(np.unique(y_train)) > 1 else np.nan
auc_ts = roc_auc_score(y_test,  y_proba_test)  if len(np.unique(y_test)) > 1  else np.nan

print(f"[9/1 XGB (mesmo split 8/1/1)] Acurácia treino: {acc_train:.4f} — teste: {acc_test:.4f}")

# ----------------- Salvando modelo, dados e métricas -----------------
# Supõe que MODEL_DIR, BASICO_CSV, BASICO_DIR, PDF_OUT, DATASET_NAME já existam como no seu fluxo original
model_name = MODEL_DIR / f'xgb_basic_ts{TEST_SIZE_PCT}_rs{RT_RANDOM_STATE}.pkl'
joblib.dump(model_basic, model_name)

# Salva X_train/X_test com índices originais preservados
X_train.to_csv(BASICO_CSV / f'X_train_xgb_basic_ts{TEST_SIZE_PCT}_rs{RT_RANDOM_STATE}.csv')
X_test.to_csv (BASICO_CSV / f'X_test_xgb_basic_ts{TEST_SIZE_PCT}_rs{RT_RANDOM_STATE}.csv')
pd.Series(y_test,  name='y_test').to_csv (BASICO_CSV / f'y_test_xgb_basic_ts{TEST_SIZE_PCT}_rs{RT_RANDOM_STATE}.csv',  index=True)
pd.Series(y_train, name='y_train').to_csv(BASICO_CSV / f'y_train_xgb_basic_ts{TEST_SIZE_PCT}_rs{RT_RANDOM_STATE}.csv', index=True)

# Salva features usadas
features = X.columns.tolist()
joblib.dump(features, MODEL_DIR / f'features_xgb_basic_ts{TEST_SIZE_PCT}_rs{RT_RANDOM_STATE}.pkl')

# Salva métricas em CSV no basico csv folder
metrics_basic = pd.DataFrame([{ 
    'model': str(model_name.name),
    'schema': '9/1 via 8/1/1 (treino=folds 1–9, teste=fold 10)',
    'test_size_alias': TEST_SIZE,   # só informativo
    'random_state': RT_RANDOM_STATE,
    'acc_train': acc_train,
    'acc_test':  acc_test,
    'f1_train':  f1_tr,
    'f1_test':   f1_ts,
    'precision_train': prec_tr,
    'precision_test':  prec_ts,
    'recall_train':    rec_tr,
    'recall_test':     rec_ts,
    'auc_train': auc_tr,
    'auc_test':  auc_ts
}])
metrics_basic.to_csv(BASICO_CSV / f'xgb_model_basic.csv', index=False)

# === Adiciona metrics_basic ao final do PDF resumo (tenta mesclar) ===
metrics_pdf_path = BASICO_DIR / f'xgb_basic_metrics_page_ts{TEST_SIZE_PCT}_rs{RT_RANDOM_STATE}.pdf'
main_pdf   = PDF_OUT / f'XGB_CV_SUMMARY_{N_SPLITS}.pdf'
merged_pdf = PDF_OUT / f'XGB_CV_SUMMARY_{N_SPLITS}_merged.pdf'
try:
    fig, ax = plt.subplots(figsize=(11.69, 8.27))  # A4 landscape
    ax.axis('off')
    display_df = metrics_basic.copy().round(4)
    table = ax.table(cellText=display_df.values, colLabels=display_df.columns, loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.2)
    ax.set_title('Tabela: Métricas - XGBoost Básico (9/1 mesmo split 8/1/1)', pad=20)
    plt.tight_layout()
    with PdfPages(metrics_pdf_path) as pdf2:
        pdf2.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    try:
        from PyPDF2 import PdfMerger
        merger = PdfMerger()
        if main_pdf.exists():
            merger.append(str(main_pdf))
        merger.append(str(metrics_pdf_path))
        with open(merged_pdf, 'wb') as f:
            merger.write(f)
        merged_pdf.replace(main_pdf)
        if metrics_pdf_path.exists():
            metrics_pdf_path.unlink()
        print('metrics_basic (XGB 9/1, mesmo split 8/1/1) adicionada ao final do PDF resumo.')
    except Exception as e_merge:
        print('Não foi possível mesclar PDFs automaticamente (PyPDF2 pode estar ausente).')
        print('O PDF com as métricas foi salvo em:', metrics_pdf_path, 'Erro:', e_merge)
except Exception as e:
    print('Não foi possível gerar a página PDF de metrics_basic:', e)

# ----------------- Dataset augmentado (df original + preds) -----------------
try:
    X_test_reset = X_test.reset_index().rename(columns={'index': 'orig_index'})
    preds_df = pd.DataFrame({
        'orig_index': X_test_reset['orig_index'],
        'y_pred': y_pred_test_bin,
        'y_proba': y_proba_test
    })
    df_orig = df.reset_index().rename(columns={'index': 'orig_index'})
    df_augmented = pd.merge(df_orig, preds_df, how='left', on='orig_index')
    df_augmented.to_csv(BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_ts{TEST_SIZE_PCT}_rs{RT_RANDOM_STATE}.csv', index=False)
    df_augmented.to_csv(CSV_OUT     / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_ts{TEST_SIZE_PCT}_rs{RT_RANDOM_STATE}.csv', index=False)
except Exception as e_aug:
    print('Não foi possível gerar dataset augmentado do basico:', e_aug)

# Seleciona exemplos aleatórios do X_test para explicação (12 por padrão)
num_instancias = 12
np.random.seed(42)
indices_aleatorios = np.random.choice(X_test.index, size=min(num_instancias, len(X_test)), replace=False)

print('Modelo salvo em:', model_name)
print('Métricas salvas em:', BASICO_CSV / f'xgb_model_basic.csv')
print('Dataset augmentado salvo em:', BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_ts{TEST_SIZE_PCT}_rs{RT_RANDOM_STATE}.csv')
print('Índices amostra aleatorios uso diverso:', indices_aleatorios)


In [ ]:
# Salvar X_test_basic_full.csv e X_train_basic_full.csv
# com todas as colunas do arquivo cru + y, y_pred, y_proba
import pandas as pd

# Carregar o arquivo cru
raw_df = pd.read_csv(DATASET_PATH)


# Função para montar e salvar o DataFrame completo (alinhando pelos índices originais)
def save_full_split(indices, y_true, y_pred, y_proba, split_name):
    idx = pd.Index(indices)
    df_full = raw_df.loc[idx].copy()

    y_true_s  = pd.Series(y_true,  index=idx)
    y_pred_s  = pd.Series(y_pred,  index=idx)
    y_proba_s = pd.Series(y_proba, index=idx)

    df_full['y']       = y_true_s.values
    df_full['y_pred']  = y_pred_s.values
    df_full['y_proba'] = y_proba_s.values

    df_full.to_csv(BASICO_CSV / f'X_{split_name}_basic_full.csv', index=True)

# Usa os índices originais já preservados em X_train / X_test (9/1 via KFold)
save_full_split(X_test.index,  y_test,  y_pred_test_bin,  y_proba_test,  'test')
save_full_split(X_train.index, y_train, y_pred_train_bin, y_proba_train, 'train')
